# limeの結果を作成する



In [ ]:
matcher_names = ["bert_mini", "magellan"]
dataset_root_dir = "../../data/lemon/datasets"
model_root_dir = "../../data/lemon/model"
out_root_dir = "../../data/experiments/10_eval/lime_result"
dataset_names = [
    "structured_amazon_google",
    "structured_beer",
    "structured_dblp_acm",
    "structured_dblp_google_scholar",
    "structured_fodors_zagat",
    "structured_walmart_amazon",
    "structured_itunes_amazon",
    "dirty_dblp_acm",
    "dirty_dblp_google_scholar",
    "dirty_walmart_amazon",
    "dirty_itunes_amazon",
    "textual_abt_buy",
    "textual_company",
]

In [ ]:
TARGET_DATASET_ID = 0
TOP_N = 5
TARGET_MATCHER_ID = None
START_DATA_IDX = None
END_DATA_IDX = None
GPU_ID = 0

In [ ]:
BATCH_SIZE = 512

In [ ]:
# torchモジュールの読み込み前に、利用できるGPUを指定しておく
## これをやらないと、システム内のＧＰＵすべてを利用してしまう
import os

os.environ["CUDA_VISIBLE_DEVICES"] = f"{GPU_ID}"

import torch

print("CUDA =", torch.cuda.is_available())
print("CUDA DEVICES =", torch.cuda.device_count())
print("CUDA CURRENT DEVICE_ID = ", torch.cuda.current_device())

In [ ]:
# transformers の tokenizer を並列実行で呼び出すか（dead lockしてしまう）
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
# Set Random Seeds and Reproducibility
import random

import numpy as np


def set_seed(seed: int):
    """
    Helper function for reproducible behavior to set the seed in ``random``, ``numpy``, ``torch``
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(0)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

In [ ]:
import pathlib
import pickle
from typing import List

import tqdm

from pine.dataset import load_dataset
from pine.matcher.magellan_matcher import make_magellan_matcher_func
from pine.matcher.transformer_matcher import make_transformer_matcher_func
from pine.entity import Entity, EntityPair
from pine.explainer import AttributionScore
from pine.explainer.lime_explainer import make_explanation, kernel


def save_lime_results(
    dataset,
    matcher_func,
    out_dir,
    save_step,
    top_n,
    start_data_idx=None,
    end_data_idx=None,
):
    for i in tqdm.tqdm(range(0, len(dataset.test.record_id_pairs), save_step)):
        # 開始idxよりも前ならskip
        if (
            start_data_idx is not None
            and i < int(start_data_idx / save_step) * save_step
        ):
            print("skip step {}".format(i))
            continue
        # 終了idxよりも後ろならskip
        if (
            end_data_idx is not None
            and int((end_data_idx - 1) / save_step) * save_step < i
        ):
            print("skip step {}".format(i))
            continue

        lime_results = {}
        out_file_path = pathlib.Path(out_dir) / f"{i}.pickle"

        if out_file_path.exists():
            # ファイルがあればデータを読み込む
            with out_file_path.open("rb") as f:
                lime_results = pickle.load(f)

        for data_count, (pid, l_id, r_id) in tqdm.tqdm(
            enumerate(
                dataset.test.record_id_pairs.iloc[i : i + save_step].itertuples()
            ),
            total=save_step,
            leave=False,
        ):
            # 開始idxよりも前ならskip
            if start_data_idx is not None and i + data_count < start_data_idx:
                print("skip {}".format(i + data_count))
                continue
            # 終了idxよりも後ろならskip
            if end_data_idx is not None and end_data_idx - 1 < i + data_count:
                print("skip {}".format(i + data_count))
                continue
            if pid not in lime_results:
                lime_results[pid] = {}

            entity_l = Entity.from_dataframe(dataset.test.records.a.loc[[l_id]])
            entity_r = Entity.from_dataframe(dataset.test.records.b.loc[[r_id]])

            # オリジナルがなければ作る
            if (None, None) not in lime_results[pid]:
                ret = make_explanation(
                    EntityPair(entity_l, entity_r), matcher_func, kernel, None
                )
                lime_results[pid][(None, None)] = ret

            # TOP K Attribution score (attribution score >0 and attribution score 降順)を削除
            attribution_score_list_l: List[AttributionScore] = lime_results[pid][
                (None, None)
            ][0]
            attribution_score_list_r: List[AttributionScore] = lime_results[pid][
                (None, None)
            ][1]
            attribution_score_list_l = sorted(
                attribution_score_list_l, key=lambda x: x.score, reverse=True
            )
            attribution_score_list_r = sorted(
                attribution_score_list_r, key=lambda x: x.score, reverse=True
            )
            del_idx_ls = []
            del_idx_rs = []
            # top n target idx
            for attribution_score in attribution_score_list_l[:top_n]:
                if attribution_score.score <= 0:
                    break
                del_idx_ls.append(attribution_score.index)
            for attribution_score in attribution_score_list_r[:top_n]:
                if attribution_score.score <= 0:
                    break
                del_idx_rs.append(attribution_score.index)
            # bottom n target idx
            for attribution_score in attribution_score_list_l[::-1][:top_n]:
                if attribution_score.score >= 0:
                    break
                del_idx_ls.append(attribution_score.index)
            for attribution_score in attribution_score_list_r[::-1][:top_n]:
                if attribution_score.score >= 0:
                    break
                del_idx_rs.append(attribution_score.index)

            # in entity_l
            for del_idx_l in del_idx_ls:
                # 未作成なら作る
                if (del_idx_l, None) not in lime_results[pid]:
                    entity_l_del = entity_l.make_entity_by_deleting_segments(
                        [del_idx_l]
                    )
                    ret = make_explanation(
                        EntityPair(entity_l_del, entity_r), matcher_func, kernel, None
                    )
                    lime_results[pid][(del_idx_l, None)] = ret
            # in entity_r
            for del_idx_r in del_idx_rs:
                # 未作成なら作る
                if (None, del_idx_r) not in lime_results[pid]:
                    entity_r_del = entity_r.make_entity_by_deleting_segments(
                        [del_idx_r]
                    )
                    ret = make_explanation(
                        EntityPair(entity_l, entity_r_del), matcher_func, kernel, None
                    )
                    lime_results[pid][(None, del_idx_r)] = ret

        with out_file_path.open("wb") as f:
           pickle.dump(lime_results, f)
        print("save {}".format(str(out_file_path)))
    return True

In [ ]:
import gc

save_step = 100

target_dataset_name = dataset_names[TARGET_DATASET_ID]
dataset = load_dataset(target_dataset_name, dataset_root_dir)
for target_matcher_name in matcher_names:
    print("=======================")
    print(target_dataset_name, target_matcher_name)
    print("=======================")
    if TARGET_MATCHER_ID is not None and target_matcher_name != matcher_names[TARGET_MATCHER_ID]:
        print("SKIP. Because TARGET_MATCHER_NAME={}".format(matcher_names[TARGET_MATCHER_ID]))
        continue
    if target_matcher_name == "magellan":
        matcher_func = make_magellan_matcher_func(
            target_dataset_name, model_root_dir
        )
    elif target_matcher_name == "bert_mini":
        matcher_func = make_transformer_matcher_func(
            target_dataset_name, model_root_dir
        )
    out_dir_path = (
        pathlib.Path(out_root_dir) / target_matcher_name / target_dataset_name
    )
    out_dir_path.mkdir(parents=True, exist_ok=True)
    save_lime_results(dataset, matcher_func, out_dir_path, save_step, TOP_N, START_DATA_IDX, END_DATA_IDX)
    del matcher_func
    gc.collect()